# **Thông tin nhóm**
- Lớp: ML Labs 23KHDL1
- Nhóm: 4
- Sinh viên: 
    - 23127102 - Lê Quang Phúc
    - 23127212 - Nguyễn Quang Đăng Khoa
    - 23127241 - Đoàn Thành Phát
    - 23127332 - Trần Tiến Cường
    - 23127442 - Trầm Hữu Nhân


# **1. Tổng quan**

# **2. Google Maps**


## 2.1 Giới thiệu

**Mục tiêu**: Thu thập dữ liệu ~2000 địa điểm tham quan tại Thành phố Hồ Chí Minh

**Phân loại**: có nhãn category là `attraction`, bao gồm: di tích, bảo tàng, công viên, chùa, nhà thờ, khu vui chơi, TTTM, chợ, spa, gym, sở thú, thư viện, rạp phim, karaoke,...

**Phương pháp tiếp cận**: Nhóm sử dụng phương pháp **Web Scraping** thông qua giả lập trình duyệt với thư viện `Selenium` để thu thập dữ liệu từ Google Maps.

**Kết quả** (`ggmaps.csv` / `ggmaps.json`):
| Cột | Kiểu dữ liệu | Mô tả |
|-----|------|-------|
| name | str | Tên địa điểm |
| address | str | Địa chỉ |
| star | float | Đánh giá sao (0-5) |
| price | int | Giá vé |
| category | str | Phân loại nhóm |
| comments | list str | Danh sách bình luận |
| url | str | Đường dẫn trên GoogleMap |

## 2.2 Thư viện

In [1]:
%pip install selenium webdriver-manager pandas tqdm ipywidgets -q

import os
import re
import json
import time
import random
import pandas as pd

from datetime import datetime, timedelta
from tqdm.notebook import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException,
    ElementClickInterceptedException, StaleElementReferenceException
)
from webdriver_manager.chrome import ChromeDriverManager

CONFIG = {
    'headless': False,              # True = chạy ẩn, False = thấy trình duyệt
    'max_places_per_query': 200,    # Số địa điểm tối đa mỗi query
    'max_reviews_per_place': 30,    # Số reviews tối đa mỗi địa điểm
    'years_back': 5,                # Lấy reviews trong N năm trở lại
    'sleep_min': 1.5,               # Delay tối thiểu (giây)
    'sleep_max': 3.5,               # Delay tối đa (giây)
    'sleep_between_places': 1,      # Delay giữa các địa điểm
    'sleep_between_queries': 2,     # Delay giữa các query
    'output_dir': 'data',           # Thư mục lưu dữ liệu
    'checkpoint_interval': 10,      # Lưu checkpoint mỗi N địa điểm
    'disable_images': False,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 2.3 Danh sách tìm kiếm

Để thu thập dữ liệu đa dạng và bao phủ rộng các loại địa điểm tham quan tại TP. Hồ Chí Minh, nhóm xây dựng một danh sách các câu truy vấn (search queries) được sử dụng để tìm kiếm trên Google Maps (có thể mở rộng).

Cách thiết kế danh sách queries:
- Tìm kiếm theo các địa điểm
- Tìm kiếm theo khu vực địa lý
- Sử dụng cả tiếng Anh - Việt

In [2]:
SEARCH_QUERIES = {
    'attraction': [
        'historical sites in Ho Chi Minh City',
        'museums in Ho Chi Minh City',
        'bảo tàng gần đây Sài Gòn',
        'di tích lịch sử gần đây Sài Gòn',

        'temples in Ho Chi Minh City',
        'pagoda in Ho Chi Minh City',
        'churches in Ho Chi Minh City',
        'chùa gần đây Sài Gòn',
        'nhà thờ gần đây Sài Gòn',

        'parks in Ho Chi Minh City',
        'gardens in Ho Chi Minh City',
        'công viên gần đây Sài Gòn',

        'amusement parks in Ho Chi Minh City',
        'water parks in Ho Chi Minh City',
        'entertainment centers in Ho Chi Minh City',
        'khu vui chơi gần đây Sài Gòn',
        'công viên nước gần đây Sài Gòn',

        'cinemas in Ho Chi Minh City',
        'theaters in Ho Chi Minh City',
        'rạp chiếu phim gần đây Sài Gòn',

        'shopping malls in Ho Chi Minh City',
        'markets in Ho Chi Minh City',
        'trung tâm thương mại gần đây Sài Gòn',
        'chợ gần đây Sài Gòn',

        'spa in Ho Chi Minh City',
        'massage in Ho Chi Minh City',
        'spa gần đây Sài Gòn',

        'gyms in Ho Chi Minh City',
        'sports centers in Ho Chi Minh City',
        'phòng gym gần đây Sài Gòn',
        'sân bóng gần đây Sài Gòn',

        'zoo in Ho Chi Minh City',
        'aquarium in Ho Chi Minh City',

        'libraries in Ho Chi Minh City',
        'thư viện gần đây Sài Gòn',

        'karaoke in Ho Chi Minh City',
        'karaoke gần đây Sài Gòn',

        'places to visit in District 1 Ho Chi Minh',
        'places to visit in District 2 Ho Chi Minh',
        'places to visit in District 3 Ho Chi Minh',
        'places to visit in District 4 Ho Chi Minh',
        'places to visit in District 5 Ho Chi Minh',
        'places to visit in District 6 Ho Chi Minh',
        'places to visit in District 7 Ho Chi Minh',
        'places to visit in District 8 Ho Chi Minh',
        'places to visit in District 9 Ho Chi Minh',
        'places to visit in District 10 Ho Chi Minh',
        'places to visit in District 11 Ho Chi Minh',
        'places to visit in District 12 Ho Chi Minh',
        'places to visit in Go Vap Ho Chi Minh',
        'places to visit in Binh Thanh Ho Chi Minh',
        'places to visit in Phu Nhuan Ho Chi Minh',
        'places to visit in Tan Phu Ho Chi Minh',
        'places to visit in Tan Binh Ho Chi Minh',
        'places to visit in Binh Tan Ho Chi Minh',
        'places to visit in Hoc Mon Ho Chi Minh',
        'places to visit in Cu Chi Ho Chi Minh',
        'places to visit in Binh Chanh Ho Chi Minh',
        'places to visit in Can Gio Ho Chi Minh',
        'places to visit in Nha Be Ho Chi Minh',
        'places to visit in Thu Duc Ho Chi Minh',

        'tourist attractions in Ho Chi Minh City',
        'things to do in Ho Chi Minh City',
        'best places to visit in Saigon',
        'top attractions Saigon',
        'điểm tham quan nổi tiếng Sài Gòn',
        'địa điểm du lịch gần đây Sài Gòn',
    ],
}


total_queries = len(SEARCH_QUERIES['attraction'])
print(f"Tổng số queries: {total_queries}")
print(f"Số lượng dữ liệu ước tính: {total_queries} queries × {CONFIG['max_places_per_query']} places = {total_queries * CONFIG['max_places_per_query']} (trước khi khử trùng lập)")

Tổng số queries: 67
Số lượng dữ liệu ước tính: 67 queries × 200 places = 13400 (trước khi khử trùng lập)


## 2.4 Thu thập dữ liệu
Nhóm khởi tạo một trình duyệt Chrome thực sự và điều khiển nó tự động thông qua việc mô phỏng hoàn toàn các thao tác của người dùng thật: nhập từ khóa tìm kiếm, cuộn danh sách kết quả, nhấn vào từng địa điểm, chuyển tab, và đọc nội dung hiển thị trên trang.

Để tránh bị Google phát hiện và chặn dưới dạng bot, nhóm áp dụng một số kỹ thuật chống phát hiện:
- Ẩn dấu hiệu tự động hóa bằng cách ghi đè thuộc tính `navigator.webdriver` về `undefined`, loại bỏ cờ `enable-automation` và tắt `useAutomationExtension`
- Thiết lập đồ trễ ngẫu nhiên giữa mỗi thao tác nhầm mô phỏng tốc độ tương tác tự nhiên của con người thay vì gửi request liên tục với tần suất cố định.
- Cài đặt cơ chế checkpoint để lưu dữ liệu định kỳ, đảm bảo không mất dữ liệu nếu quá trình thu thập bị gián đoạn do lỗi mạng, bị chặn, hoặc người dùng dừng thủ công.

## 2.4.1 Khai báo lớp

Lớp `GoogleMapsScraper` bao gồm các nhóm phương thức chính: **Khởi tạo & Tiện ích**, **Tìm kiếm & Cuộn**, **Trích xuất dữ liệu** và **Lưu trữ**

- Khởi tạo và tiện ích:
    - Nhận cầu hình từ `CONFIG`, khởi tạo danh sách để lưu trữ kể quả và chuẩn bị trình duyệt
    - Cấu hình và khởi tạo trình duyệt với các thiết lập: tự động hóa, tùy chọn hiệu năng, quản lý phiên bản, chấp nhận cookie

- Tìm kiếm và cuộn danh sách: 
    - Xây dựng URL tìm kiếm với các query đã mã hóa với tọa độ trong tâm Thành phố Hồ Chí Minh (mức zoom 12)
    - Cuộn danh sách kết quả để tải thêm địa điểm đến khi đạt đủ số lượng hoặc đến cuối danh sách
    - Tạo thành danh sách URL với mỗi đường dẫn tương ứng với một địa điểm trên Google Maps
- Trích xuất dữ liệu:
    - Mã định danh duy nhất dùng cho việc khử trùng lặp 
    - `name`: tên địa điểm từ thẻ tiêu đề
    - `star`: sao đánh giá 
    - `price`: giá vé vào cửa (đã quy sang VND), được trích xuất với 3 cách tiếp cận khác nhau
    - `address`: địa chỉ, được trích xuất với 5 cách tiếp cận khác nhau  
    - `category`: nhãn phân loại (được gán nhãn `attraction`)
    - `comments`: danh sách bình luận
    - `url`: đường dẫn Google Maps 
- Lưu trữ: 
    - Lưu dữ liệu tạm vào file checkpoint.csv sau `N` địa điểm
    - Lưu kết quả cuối cùng ở 2 định dạng: **.csv** (`ggmap.csv`) dùng cho phân tích dạng bảng và **.json** (`ggmap.json`) dùng cho trao đổi dữ liệu

In [ ]:
COLUMN_ORDER = ['name', 'address', 'star', 'price', 'category', 'comments', 'url']

class GoogleMapsScraper:
    def __init__(self, config=None):
        self.config = config or CONFIG
        self.all_data = []
        self.seen_place_ids = set()
        self._setup_driver()

    # Khởi tạo và tiện ích
    def _setup_driver(self):
        options = Options()
        options.add_argument('--lang=vi')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_argument('--window-size=1920,1080')
        
        if self.config.get('disable_images', True):
            prefs = {"profile.managed_default_content_settings.images": 2}
            options.add_experimental_option("prefs", prefs)
        
        options.add_argument('--disable-gpu') 
        options.add_argument('--disable-extensions')
        options.add_argument('--dns-prefetch-disable')
        options.add_experimental_option('excludeSwitches', ['enable-automation'])
        options.add_experimental_option('useAutomationExtension', False)

        if self.config.get('headless', False):
            options.add_argument('--headless=new')

        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=options)
        
        self.driver.execute_script(
            "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        )
        
        self.wait = WebDriverWait(self.driver, 10)

    def _random_sleep(self, min_sec=None, max_sec=None):
        lo = min_sec or self.config['sleep_min']
        hi = max_sec or self.config['sleep_max']
        time.sleep(random.uniform(lo, hi))

    def _accept_cookies(self):
        selectors = [
            'button[aria-label*="Accept"]',
            'button[aria-label*="Chấp nhận"]',
            'button[aria-label*="Đồng ý"]',
            'form[action*="consent"] button',
            'button[jsname="b3VHJd"]',
        ]
        for sel in selectors:
            try:
                btn = WebDriverWait(self.driver, 3).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, sel))
                )
                btn.click()
                time.sleep(1)
                return
            except TimeoutException:
                continue

    # Tìm kiếm và cuộn
    def search_places(self, query):
        import urllib.parse

        encoded_query = urllib.parse.quote(query)
        search_url = (
            f"https://www.google.com/maps/search/{encoded_query}/"
            f"@10.8231,106.6297,12z"
        )
        
        self.driver.get(search_url)
        self._random_sleep(1, 2)
        self._accept_cookies()

        # Kiểm tra có phải là 1 địa điểm cụ thể hay không (không phải danh sách kết quả)
        current_url = self.driver.current_url
        if '/place/' in current_url and '/search/' not in current_url:
            print(f"    [INFO] Redirected to single place, trying search box...")
            try:
                search_box = WebDriverWait(self.driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR,
                        'input#searchboxinput, input[name="q"], '
                        'input[aria-label*="Search"], input[aria-label*="Tìm kiếm"]'))
                )
                search_box.clear()
                time.sleep(0.5)
                search_box.send_keys(query)
                time.sleep(0.5)
                search_box.send_keys(Keys.ENTER)
                self._random_sleep(2, 3)
            except Exception as e:
                print(f"    [WARN] Search box fallback failed: {e}")

        # Hiển thị danh sách kết quả
        try:
            WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="feed"]'))
            )
            return True
        except TimeoutException:
            print(f"Không thấy danh sách kết quả cho: {query}")
            try:
                WebDriverWait(self.driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="feed"]'))
                )
                return True
            except TimeoutException:
                print(f"Bỏ qua query: {query}")
                return False

    def scroll_results(self, target_count=None):
        if target_count is None:
            target_count = self.config['max_places_per_query']

        try:
            # Xác định vùng cuộn 
            feed = self.wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="feed"]'))
            )
        except TimeoutException:
            print("Không tìm thấy results feed")
            return 0

        last_count = 0
        no_change_count = 0

        # Thực hiện cuốn để đếm số lượng dữ liệu
        while no_change_count < 5:
            self.driver.execute_script(
                'arguments[0].scrollTop = arguments[0].scrollHeight', feed
            )
            self._random_sleep(0.5, 1)

            results = feed.find_elements(By.CSS_SELECTOR, 'a.hfpxzc')
            current_count = len(results)

            if current_count >= target_count:
                break

            if current_count == last_count:
                no_change_count += 1
                try:
                    end_el = self.driver.find_element(By.XPATH,
                        '//span[contains(text(),"end of the list") or '
                        'contains(text(),"cuối danh sách") or '
                        'contains(text(),"xem hết")]')
                    if end_el.is_displayed():
                        break
                except NoSuchElementException:
                    pass
            else:
                no_change_count = 0
            last_count = current_count

        final = feed.find_elements(By.CSS_SELECTOR, 'a.hfpxzc')

        return len(final)

    def get_place_urls(self):
        urls = []

        try:
            feed = self.driver.find_element(By.CSS_SELECTOR, 'div[role="feed"]')
            links = feed.find_elements(By.CSS_SELECTOR, 'a.hfpxzc')

            for link in links:
                href = link.get_attribute('href')
                if href:
                    urls.append(href)
        except Exception as e:
            print(f"Lỗi lấy URLs: {e}")
            
        return urls

    # Trích xuất dữ liệu
    def _safe_text(self, by, selector, default=''):
        try:
            return self.driver.find_element(by, selector).text.strip()
        except (NoSuchElementException, StaleElementReferenceException):
            return default

    def _safe_attr(self, by, selector, attr, default=''):
        try:
            return self.driver.find_element(by, selector).get_attribute(attr)
        except (NoSuchElementException, StaleElementReferenceException):
            return default

    def _extract_place_id(self, url):
        # ID nội bộ của GoogleMap
        match = re.search(r'!1s(0x[a-f0-9]+:0x[a-f0-9]+)', url)
        if match:
            return match.group(1)
        
        # ID trên url
        match = re.search(r'/place/([^/]+)/', url)
        if match:
            return match.group(1)
        
        # Giữ nguyên
        return url

    def extract_place(self, url, pid):
        place_id = pid

        try:
            self.driver.get(url)
            self._random_sleep(1, 2)

            record = {}

            # Name (str)
            record['name'] = self._safe_text(By.CSS_SELECTOR, 'h1.DUwDvf')
            if not record['name']:
                record['name'] = self._safe_text(By.CSS_SELECTOR, 'h1')

            # Star (float)
            rating_text = self._safe_text(
                By.CSS_SELECTOR, 'div.F7nice span[aria-hidden="true"]'
            )
            try:
                record['star'] = float(rating_text.replace(',', '.'))
            except (ValueError, AttributeError):
                record['star'] = None

            # Price (int)
            record['price'] = self._extract_price()

            # Address (str)
            record['address'] = self._extract_address()

            # Category (str)
            record['category'] = 'attraction'

            # Comments (list of str)
            record['comments'] = self._get_comments(url, place_id)

            # URL (str)
            record['url'] = url

            self.seen_place_ids.add(place_id)

            return record

        except Exception as e:
            print(f"Lỗi trong việc extract_place: {e}")
            return None

    def _extract_address(self):     
        '''
        Đảm bảo ở trong tab Tổng quan
        '''
        # Cách 1: Các selector phổ biến nhất
        address_selectors = [
            (By.CSS_SELECTOR, 'button[data-item-id="address"] div.Io6YTe'),
            (By.CSS_SELECTOR, 'button[data-item-id="address"] div.fontBodyMedium'),
            (By.CSS_SELECTOR, 'button[data-item-id="oloc"] div.Io6YTe'),
            (By.CSS_SELECTOR, 'button[data-item-id="oloc"] div.fontBodyMedium'),
            (By.CSS_SELECTOR, 'button[data-item-id*="laddress"] div.Io6YTe'),
            (By.CSS_SELECTOR, 'button[data-item-id^="address"] div.Io6YTe'),
        ]
        for by, sel in address_selectors:
            try:
                el = self.driver.find_element(by, sel)
                text = el.text.strip()
                if text:
                    return text
            except (NoSuchElementException, StaleElementReferenceException):
                continue

        # Cách 2: Tìm qua aria-label chứa "Địa chỉ"
        try:
            el = self.driver.find_element(By.XPATH,
                '//button[contains(@aria-label,"Địa chỉ:") or contains(@aria-label,"Address:")]')
            aria = el.get_attribute('aria-label')
            if aria:
                for prefix in ['Địa chỉ:', 'Address:']:
                    if prefix in aria:
                        addr = aria.split(prefix, 1)[1].strip()
                        if addr:
                            return addr
        except (NoSuchElementException, StaleElementReferenceException):
            pass

        # Cách 3: Tìm icon location pin -> lấy text
        try:
            addr_el = self.driver.find_element(By.XPATH,
                '//button[@data-tooltip and @data-tooltip="Sao chép địa chỉ"]//div[contains(@class,"Io6YTe")]'
                ' | //button[@data-tooltip and @data-tooltip="Copy address"]//div[contains(@class,"Io6YTe")]')
            text = addr_el.text.strip()
            if text:
                return text
        except (NoSuchElementException, StaleElementReferenceException):
            pass

        # Cách 4: Scroll xuống để tìm address (có thể bị ẩn ở dưới)
        try:
            scrollable = self.driver.find_element(By.CSS_SELECTOR, 'div.m6QErb.DxyBCb.kA9KIf.dS8AEf')
            self.driver.execute_script(
                'arguments[0].scrollTop = arguments[0].scrollTop + 300', scrollable
            )
            self._random_sleep(0, 0.5)

            for by, sel in address_selectors[:2]:
                try:
                    el = self.driver.find_element(by, sel)
                    text = el.text.strip()
                    if text:
                        return text
                except (NoSuchElementException, StaleElementReferenceException):
                    continue
        except (NoSuchElementException, StaleElementReferenceException):
            pass

        # Cách 5: Lấy tất cả button có class "CsEnBe" chứa div.Io6YTe — tìm dòng có dấu hiệu là địa chỉ
        try:
            buttons = self.driver.find_elements(By.CSS_SELECTOR, 'button.CsEnBe')
            for btn in buttons:
                try:
                    io6 = btn.find_element(By.CSS_SELECTOR, 'div.Io6YTe')
                    text = io6.text.strip()
                    addr_keywords = ['Quận', 'District', 'Phường', 'Ward', 'Đường',
                                     'Street', 'TP', 'Thành phố', 'Ho Chi Minh',
                                     'Hồ Chí Minh', 'Việt Nam', 'Vietnam']
                    if text and any(kw.lower() in text.lower() for kw in addr_keywords):
                        return text
                except NoSuchElementException:
                    continue
        except Exception:
            pass

        print(f"Không tìm thấy địa chỉ")
        return ''

    def _parse_price_text(self, text):
        # Định dạng USD
        usd_patterns = [
            r'(\d+[.,]\d+)\s*US\$',                  
            r'(\d+[.,]\d+)\s*(?:USD|\$)',             
            r'(?:US\$|USD|\$)\s*(\d+[.,]\d+)',       
            r'(\d+)\s*US\$',                          
            r'(\d+)\s*(?:USD|\$)',                     
            r'(?:US\$|USD|\$)\s*(\d+)',               
        ]
        
        # Định dạng VND
        vnd_patterns = [
            r'(\d{1,3}(?:\.\d{3})+)\s*(?:đ|đồng|VND|VNĐ|vnđ|₫)',
            r'(\d[\d\.]*)\s*(?:đ|đồng|VND|VNĐ|vnđ|₫)',
            r'(?:giá vé|ticket|admission|entrance fee|vé vào cửa)[:\s]*(\d[\d\.]*)',
        ]

        for pattern in usd_patterns:
            m = re.search(pattern, text, re.IGNORECASE)
            if m:
                price_str = m.group(1).replace(',', '.')
                try:
                    usd_val = float(price_str)
                    if 0.01 <= usd_val <= 1000:
                        result = int(usd_val * 26000)
                        return result
                except ValueError:
                    continue

        for pattern in vnd_patterns:
            m = re.search(pattern, text, re.IGNORECASE)
            if m:
                price_str = m.group(1).replace('.', '')
                try:
                    price_val = int(price_str)
                    if 1000 <= price_val <= 10_000_000:
                        return price_val
                except ValueError:
                    continue

        return None

    def _extract_price(self):
        # Cách 1: Tìm trong tab Tổng quan (Overview)
        ticket_xpaths = [
            '//div[contains(text(),"Vé vào cửa") or contains(text(),"Admission") or contains(text(),"Tickets")]/..',
            '//h2[contains(text(),"Vé vào cửa") or contains(text(),"Admission") or contains(text(),"Tickets")]/..',
            '//div[contains(@aria-label,"Vé vào cửa") or contains(@aria-label,"ticket") or contains(@aria-label,"Admission")]',
        ]

        for xpath in ticket_xpaths:
            try:
                section = self.driver.find_element(By.XPATH, xpath)
                text = section.text
                price = self._parse_price_text(text)
                if price:
                    return price
            except NoSuchElementException:
                continue

        # Cách 2: Tìm trong tab Vé
        try:
            ticket_tab = None
            tab_selectors = [
                (By.CSS_SELECTOR, "button[aria-label*='Vé']"),
                (By.CSS_SELECTOR, "button[aria-label*='Tickets']"),
                (By.CSS_SELECTOR, "div[role='tab'][aria-label*='Vé']"),
                (By.CSS_SELECTOR, "div[role='tab'][aria-label*='Tickets']"),
                (By.XPATH, '//button[.//div[contains(text(),"Vé")] or .//div[contains(text(),"Tickets")]]'),
                (By.XPATH, '//div[@role="tab" and (contains(.,"Vé") or contains(.,"Tickets"))]'),
                (By.XPATH, '//div[@role="tablist"]//button[contains(.,"Vé") or contains(.,"Tickets")]'),
                (By.XPATH, '//*[self::button or self::div[@role="tab"]][.//text()[normalize-space()="Vé"]]'),
            ]

            for by, sel in tab_selectors:
                try:
                    ticket_tab = self.driver.find_element(by, sel)
                    if ticket_tab.is_displayed():
                        break
                    ticket_tab = None
                except NoSuchElementException:
                    continue

            if ticket_tab:
                self.driver.execute_script("arguments[0].click();", ticket_tab)
                self._random_sleep(1, 2)

                price_found = None

                # Cách 2a: Ưu tiên giá vé của trang web chính thức
                try:
                    price_elements = self.driver.find_elements(By.XPATH,
                        '//*[contains(text(),"US$") or contains(text(),"USD") '
                        'or contains(text(),"₫") or contains(text(),"VND") '
                        'or contains(text(),"đ")]'
                    )
                    for el in price_elements:
                        el_text = el.text.strip()
                        if not el_text:
                            continue
                        price = self._parse_price_text(el_text)
                        if price:
                            price_found = price
                            break  
                except Exception:
                    pass

                # Cách 2b: Nếu không tìm được giá vé trong các element nhỏ, thử tìm trong các container lớn hơn
                if not price_found:
                    content_selectors = [
                        'div.m6QErb.DxyBCb',
                        'div.m6QErb',
                        'div[role="main"]',
                    ]
                    for sel in content_selectors:
                        try:
                            els = self.driver.find_elements(By.CSS_SELECTOR, sel)
                            for el in els:
                                el_text = el.text
                                if not el_text:
                                    continue
                                price = self._parse_price_text(el_text)
                                if price:
                                    price_found = price
                                    break
                            if price_found:
                                break
                        except NoSuchElementException:
                            continue

                # Quay lại tab Tổng quan -> để thực hiện việc lấy các thuộc tính khác
                try:
                    overview_tab = self.driver.find_element(By.XPATH,
                        '//button[.//div[contains(text(),"Tổng quan") or contains(text(),"Overview")]] | '
                        '//div[@role="tab" and (contains(.,"Tổng quan") or contains(.,"Overview"))]')
                    self.driver.execute_script("arguments[0].click();", overview_tab)
                    self._random_sleep(0.5, 1)
                except NoSuchElementException:
                    pass

                if price_found:
                    return price_found
                else:
                    print(f"Không thu thập được giá vé")
            else:
                print(f"Không thu thập được giá vé")

        except Exception as e:
            print(f"Lỗi trong việc lấy giá vé: {e}")

        # Cách 3: Tìm trong tab Giới thiệu 
        about_selectors = [
            'div.WeS02d.fontBodyMedium',
            'div[class*="PYvSYb"]',
            'div.rogA2c',
        ]
        for sel in about_selectors:
            try:
                el = self.driver.find_element(By.CSS_SELECTOR, sel)
                price = self._parse_price_text(el.text)
                if price:
                    print(f"    [PRICE] Found in about section: {price} VNĐ")
                    return price
            except NoSuchElementException:
                continue

        return None

    def _get_comments(self, place_url, place_id):
        max_reviews = self.config['max_reviews_per_place']
        comments = []

        try:
            reviews_tab = self._find_reviews_tab()
            if not reviews_tab:
                return comments

            reviews_tab.click()
            self._random_sleep(2, 3)

            self._sort_reviews_newest()
            self._scroll_reviews(max_reviews)

            review_elements = self.driver.find_elements(By.CSS_SELECTOR, 'div.jftiEf')

            for elem in review_elements[:max_reviews]:
                try:
                    try:
                        more_btn = elem.find_element(By.CSS_SELECTOR, 'button.w8nwRe.kyuRq')
                        more_btn.click()
                        time.sleep(0.3)
                    except (NoSuchElementException, ElementClickInterceptedException):
                        pass

                    try:
                        review_time = elem.find_element(
                            By.CSS_SELECTOR, 'span.rsqaWe'
                        ).text.strip()
                    except NoSuchElementException:
                        review_time = ''

                    if not self._is_within_timeframe(review_time):
                        continue

                    try:
                        text = elem.find_element(
                            By.CSS_SELECTOR, 'span.wiI7pd'
                        ).text.strip()
                        text = re.sub(r'\s+', ' ', text).strip()
                        if text:
                            comments.append(text)
                    except NoSuchElementException:
                        pass

                except Exception:
                    continue

        except Exception as e:
            print(f"Lỗi trong việc lấy comments: {e}")

        return comments

    def _find_reviews_tab(self):
        for sel in [
            "button[aria-label*='Reviews']",
            "button[aria-label*='Bài đánh giá']",
            "button[aria-label*='đánh giá']",
        ]:
            try:
                return self.driver.find_element(By.CSS_SELECTOR, sel)
            except NoSuchElementException:
                continue
        try:
            return self.driver.find_element(By.XPATH,
                '//button[.//div[contains(text(),"Reviews") or '
                'contains(text(),"Đánh giá") or '
                'contains(text(),"Bài đánh giá")]]')
        except NoSuchElementException:
            return None

    def _sort_reviews_newest(self):
        try:
            sort_btn = self.driver.find_element(By.XPATH,
                '//button[@aria-label="Sort reviews" or '
                '@aria-label="Sắp xếp bài đánh giá" or '
                'contains(@aria-label,"Sort") or '
                'contains(@data-value,"sort")]')
            sort_btn.click()
            self._random_sleep(1, 2)

            newest = self.driver.find_element(By.XPATH,
                '//div[@role="menuitemradio" and '
                '(@data-index="1" or contains(.,"Newest") or contains(.,"Mới nhất"))]')
            newest.click()
            self._random_sleep(1, 2)
        except (NoSuchElementException, ElementClickInterceptedException):
            pass

    def _scroll_reviews(self, target):
        scrollable = None
        for sel in [
            'div.m6QErb.DxyBCb.kA9KIf.dS8AEf',
            'div.m6QErb.DxyBCb',
        ]:
            try:
                scrollable = self.driver.find_element(By.CSS_SELECTOR, sel)
                break
            except NoSuchElementException:
                continue
        if not scrollable:
            return

        last_count, no_change = 0, 0
        while no_change < 3:
            self.driver.execute_script(
                'arguments[0].scrollTop = arguments[0].scrollHeight', scrollable
            )
            self._random_sleep(1, 2)
            elems = self.driver.find_elements(By.CSS_SELECTOR, 'div.jftiEf')
            if len(elems) >= target:
                break
            if len(elems) == last_count:
                no_change += 1
            else:
                no_change = 0
            last_count = len(elems)

    def _is_within_timeframe(self, time_text):
        if not time_text:
            return True
        txt = time_text.lower()
        year_match = re.search(r'(\d+)\s*(năm|year)', txt)
        if year_match:
            return int(year_match.group(1)) <= self.config['years_back']
        if any(w in txt for w in [
            'tháng', 'tuần', 'ngày', 'giờ', 'phút',
            'month', 'week', 'day', 'hour', 'minute'
        ]):
            return True
        return True

    # Main loop
    def scrape(self, queries, pbar=None):
        for i, query in enumerate(queries):
            try:
                found = self.search_places(query)
                if not found:
                    continue

                num = self.scroll_results()
                urls = self.get_place_urls()

                for url in urls:
                    pid = self._extract_place_id(url)
                    if pid in self.seen_place_ids:
                        continue

                    # Thu thập được tất cả thuộc tính cho 1 địa điểm
                    record = self.extract_place(url, pid)
                    if record:
                        self.all_data.append(record)

                        if pbar:
                            pbar.update(1)
                            pbar.set_postfix({
                                'total': len(self.all_data),
                                'last': record['name'][:20],
                            })

                        # Khi đạt đủ checkpoint_interval địa điểm, lưu lại dữ liệu
                        if len(self.all_data) % self.config['checkpoint_interval'] == 0:
                            self._save_checkpoint()

                    self._random_sleep(
                        self.config['sleep_between_places'],
                        self.config['sleep_between_places'] + 2
                    )

            except Exception as e:
                print(f"Lỗi khi thực hiện tìm kiếm '{query}': {e}")
                continue

            self._random_sleep(
                self.config['sleep_between_queries'],
                self.config['sleep_between_queries'] + 2
            )
            
    @staticmethod
    def _format_comments(comments_list):
        if not comments_list or not isinstance(comments_list, list):
            return '[]'
        parts = ', '.join(f'{{{c}}}' for c in comments_list)

        return f'[{parts}]'

    # Lưu trữ
    def _save_checkpoint(self):
        try:
            self._save_to_csv(
                os.path.join(self.config['output_dir'], 'checkpoint.csv')
            )
        except Exception as e:
            print(f"Lỗi khi lưu checkpoint: {e}")

    def _save_to_csv(self, filepath):
        df = pd.DataFrame(self.all_data)
        if 'comments' in df.columns:
            df['comments'] = df['comments'].apply(self._format_comments)
        
        # Đảm bảo thứ tự cột khi lưu CSV
        cols = [c for c in COLUMN_ORDER if c in df.columns]
        df = df[cols]
        df.to_csv(filepath, index=False, encoding='utf-8-sig')

    def save_final(self):
        output_dir = self.config['output_dir']

        # Định dạng .csv
        csv_path = os.path.join(output_dir, 'ggmap.csv')
        self._save_to_csv(csv_path)

        # Định dạng .json
        json_path = os.path.join(output_dir, 'ggmap.json')
        
        json_data = []
        for record in self.all_data:
            r = dict(record)
            r['comments'] = self._format_comments(r.get('comments', []))
            json_data.append(r)
        
        ordered_data = [{k: r.get(k) for k in COLUMN_ORDER} for r in json_data]
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(ordered_data, f, ensure_ascii=False, indent=2)

        print(f"Dữ liệu đã được lưu vào: {csv_path} và {json_path}")

    def close(self):
        try:
            self.driver.quit()
        except Exception:
            pass


## 2.4.2 Quá trình thực hiện

Quá trình thu thập dữ liệu được thực hiện hoàn toàn tự động thông qua việc mô phỏng hành vi của một người dùng thật đang sử dụng Google Maps để tìm kiếm và khám phá các địa điểm tham quan tại TP. Hồ Chí Minh. Quy trình gồm 4 bước chính lặp lại cho từng câu truy vấn:
- Bước 1: Tìm kiếm từ khóa
- Bước 2: Duyệt qua danh sách kết quả
- Bước 3: Thu thập thông tin từng địa điểm
- Bước 4: Lưu trữ dữ liệu

In [ ]:
scraper = GoogleMapsScraper(config=CONFIG)

queries = SEARCH_QUERIES['attraction']
estimated_total = len(queries) * CONFIG['max_places_per_query']

try:
    with tqdm(total=estimated_total, desc="Scraping") as pbar:
        scraper.scrape(queries, pbar=pbar)

    print(f"Thu thập được: {len(scraper.all_data)} điểm dữ liệu")

except KeyboardInterrupt:
    print(f"\nNgừng scraping do người dùng dừng.")
    print(f"Thu thập được: {len(scraper.all_data)} điểm dữ liệu")

    scraper._save_checkpoint()

except Exception as e:
    print(f"\nLỗi khi thực hiện scraping: {e}")

    scraper._save_checkpoint()
finally:
    scraper.close()

scraper.save_final()


Scraping:   0%|          | 0/13400 [00:00<?, ?it/s]

Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
Không thu thập được giá vé
K

# **3. Booking**
